# BPMN + Camunda: оркестрация процессов

В ноутбуке демонстрируется полный цикл: deploy BPMN -> start process -> polling состояния -> чтение истории и переменных.

## Что поднимается
- `camunda` -> :8080 (BPMN engine, REST API + Cockpit)
- `orchestrator` -> :8001 (тонкая обёртка над Camunda REST)
- `controller` -> :8002 (external-task worker, polling topic `do-work`)

```bash
docker compose up -d
sleep 60  # Camunda стартует медленно
```


In [1]:
import json, time, uuid
from datetime import datetime, timezone
import requests
print(f"requests={requests.__version__}")


requests=2.32.5


## 1. BPMN-схема процесса


In [ ]:
ORCH = "http://localhost:8001"
CAM  = "http://localhost:8080"
CKPT = f"{CAM}/camunda"

# Health check
r = requests.get(f"{ORCH}/health", timeout=5); r.raise_for_status()
print("orchestrator:", json.dumps(r.json(), indent=2))
assert r.json()["camunda_reachable"], "Camunda not reachable via orchestrator"

# Pull the BPMN XML
r = requests.get(f"{ORCH}/bpmn", timeout=5); r.raise_for_status()
bpmn_xml = r.text
print(f"BPMN length: {len(bpmn_xml)} chars\n")
print(bpmn_xml)


ConnectionError: HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8001): Failed to establish a new connection: [Errno 111] Connection refused"))

## 2. Deploy BPMN


In [ ]:
r = requests.post(f"{ORCH}/deploy", timeout=15); r.raise_for_status()
deploy = r.json()
print(json.dumps(deploy, indent=2))
assert "deployed_process_definitions" in deploy, "Deployment didn't include process definitions"
assert "demo-process" in deploy["deployed_process_definitions"], "demo-process not deployed"
DEPLOYMENT_ID = deploy["id"]
PROCESS_DEF_ID = deploy["deployed_process_definitions"]["demo-process"]["id"]
print(f"DEPLOYMENT_ID: {DEPLOYMENT_ID}")
print(f"PROCESS_DEF_ID: {PROCESS_DEF_ID}")


## 3. Запуск процесса с переменными


In [ ]:
payload = {
  "businessKey": f"demo-{uuid.uuid4().hex[:8]}",
  "variables": {
    "data":      {"value": "hello-camunda", "type": "String"},
    "quantity":  {"value": 42, "type": "Integer"},
    "is_demo":   {"value": True, "type": "Boolean"},
  },
}
# Внешняя задача с camunda:topic="do-work" будет подхвачена controller-ом (external-task worker)
r = requests.post(f"{ORCH}/start", params={"key": "demo-process"}, json=payload, timeout=10)
r.raise_for_status()
start = r.json()
print(json.dumps(start, indent=2))
PID = start["process_instance_id"]
print(f"Process instance: {PID}")


## 4. Polling состояния процесса

Контроллер опрашивает Camunda каждые 2s и выполняет service-task за ~1.5s, поэтому завершение обычно наступает через 3-7 секунд после старта.


In [ ]:
DEADLINE_SECONDS = 30
started = time.time()
states = []
while True:
    r = requests.get(f"{ORCH}/process/{PID}", timeout=5)
    r.raise_for_status()
    state = r.json()
    ended = state.get("ended", False) or not state.get("activity_instances")
    active = [a.get("activityName", a.get("activityId")) for a in state.get("activity_instances", [])]
    states.append({"t": round(time.time()-started, 1), "active": active, "ended": ended})
    print(f"  t={round(time.time()-started,1):>4.1f}s  active={active}  ended={ended}")
    if ended:
        break
    if time.time() - started > DEADLINE_SECONDS:
        print("DEADLINE EXCEEDED")
        break
    time.sleep(1.0)
print(f"\nprocess completed in {round(time.time()-started,1)}s")


## 5. История и переменные


In [ ]:
r = requests.get(f"{ORCH}/process/{PID}/history", timeout=5); r.raise_for_status()
hist = r.json()
print("activity history:")
for a in hist:
    print(f"  {a.get('activityName',a.get('activityId')):<14} start={a.get('startTime')}  end={a.get('endTime')}")

r = requests.get(f"{ORCH}/process/{PID}/variables", timeout=5); r.raise_for_status()
vars_ = r.json()
print("\nprocess variables:")
for name, v in vars_.items():
    print(f"  {name:<14} = {v.get('value')!r:<25} type={v.get('type')}")
assert "result" in vars_, "Controller did not set 'result' variable"
assert vars_["result"]["value"] == "ok"
assert "processed_by" in vars_
assert "processed_at" in vars_
print("\nOK: worker-set variables present")


## 6. Camunda Cockpit


In [ ]:
print(f"Camunda Cockpit (UI):  {CKPT}")
print(f"Login:                  demo / demo")
print(f"Process instance URL:   {CKPT}/app/cockpit/default/#/process-instance/{PID}")
print(f"Process definition:     {CKPT}/app/cockpit/default/#/process-definition/{PROCESS_DEF_ID}")
print(f"Orchestrator API:       {ORCH}")
print(f"Controller stats:       http://localhost:8002/stats")


In [ ]:
summary = """
## Итог

Мы прошли полный цикл BPMN-процесса в Camunda 7:
1. Orchestrator развернул .bpmn с external service-task в Camunda REST API.
2. Стартовали процесс с тремя переменными (data, quantity, is_demo).
3. Controller (external-task worker) подобрал задачу, симулировал работу 1.5s и установил переменные.
4. Процесс завершился. История показывает StartEvent -> ServiceTask -> EndEvent.

Стек масштабируется: добавляйте workers для других topics, ставьте таймеры и шлюзы, публикуйте сигналы через REST.
"""
print(summary)
